In [ ]:
import sys
import os
import pandas as pd
import getpass
import torch

from google.colab import userdata
openai_key = userdata.get('OPENAI_API_KEY')

In [ ]:
!git clone https://github.com/bert-tweeteval

# Add src to path
sys.path.append(os.path.abspath('src'))

from download import download_and_split_dataset
from llm_eval import LLMEvaluator, PROMPT_1_MINIMAL, PROMPT_2_STRUCTURED, LABELS

In [ ]:
# Load dataset
train_df, val_df, test_df = download_and_split_dataset()
print(f"Test set size: {len(test_df)}")

## 1. OpenAI GPT-4o-mini Evaluation

We use the `gpt-4o-mini` model with temperature 0 for deterministic results.

In [ ]:
evaluator_openai = LLMEvaluator(openai_api_key=openai_key)

print("Running GPT-4o-mini with Minimal Prompt...")
res_gpt_p1 = evaluator_openai.evaluate(test_subset, "openai", PROMPT_1_MINIMAL, num_samples=None)

print("Running GPT-4o-mini with Structured Prompt...")
res_gpt_p2 = evaluator_openai.evaluate(test_subset, "openai", PROMPT_2_STRUCTURED, num_samples=None)

## 2. Qwen3.5-0.8B Evaluation

We run the open-source Qwen model locally.

In [ ]:
evaluator_qwen = LLMEvaluator(hf_model_name="Qwen/Qwen3.5-0.8B")
evaluator_qwen.load_hf_model()

print("Running Qwen with Minimal Prompt...")
res_qwen_p1 = evaluator_qwen.evaluate(test_subset, "hf", PROMPT_1_MINIMAL, num_samples=None)

print("Running Qwen with Structured Prompt...")
res_qwen_p2 = evaluator_qwen.evaluate(test_subset, "hf", PROMPT_2_STRUCTURED, num_samples=None)

## 3. Results Comparison

We compare Accuracy, Macro-F1, and Inference Time (per 100 samples).

In [ ]:
results = {
    "GPT-4o-mini (Minimal)": res_gpt_p1,
    "GPT-4o-mini (Structured)": res_gpt_p2,
    "Qwen-0.8B (Minimal)": res_qwen_p1,
    "Qwen-0.8B (Structured)": res_qwen_p2
}

comparison_df = pd.DataFrame({
    k: {m: v[m] for m in ["Accuracy", "Macro-F1", "Time_per_100"]}
    for k, v in results.items()
}).transpose()

comparison_df.style.highlight_max(subset=["Accuracy", "Macro-F1"], color='lightgreen')